# 15 — Pending conversation: continue, stop, or switch
Apply the conversation patch on top of the V2 integration patch. Restart the old notebook kernel and leave it idle to free the GPU. Select the Agent1 `.venv` kernel here.

This is the small conversation milestone, not another benchmark or training run. Setup is copied from notebook 14 so this notebook runs independently. The base model classifies pending replies; the V2 adapter still generates grounded final answers. A bare order ID takes the fast continuation path. Other replies require an additional analysis call.

Run the setup, then the four checks. A false `expected_path` result indicates an analyzer mismatch to inspect; it is not silently counted as a pass.

In [1]:
from pathlib import Path
import sys
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/agent1").is_dir()), None)
if project_root is None:
    raise FileNotFoundError("Open the notebook inside your Agent1 project.")
sys.path.insert(0, str(project_root / "src"))
adapter_dir = project_root / "models/adapter_v2"
knowledge_path = project_root / "data/knowledge/demo_retail_knowledge_v1.json"
# Set this relative path only if you have multiple saved classifier files.
CLASSIFIER_PATH = None
if CLASSIFIER_PATH is None:
    candidates = sorted({*project_root.joinpath("models").rglob("*.joblib"), *project_root.joinpath("models").rglob("*.pkl")})
    if len(candidates) != 1:
        print("Saved model candidates:")
        for path in candidates: print(path.relative_to(project_root))
        raise ValueError("Set CLASSIFIER_PATH to your saved ensemble's relative path, then rerun this cell. No retraining is needed.")
    classifier_path = candidates[0]
else:
    classifier_path = project_root / CLASSIFIER_PATH
print("Classifier:", classifier_path)
print("Adapter:", adapter_dir)

Classifier: d:\Projects\projects\agent1\models\intent_classifier\ensemble.joblib
Adapter: d:\Projects\projects\agent1\models\adapter_v2


In [2]:
from agent1.classification.ensemble import IntentClassifierEnsemble
from agent1.routing.intent_router import IntentRouter
from agent1.routing.intent_mapping import INTENT_TO_ACTION
from agent1.analysis.request_analyzer import RequestAnalyzer
from agent1.conversation.context_builder import ContextBuilder
from agent1.rag.knowledge import KnowledgeStore
from agent1.generation.response_generator import ResponseGenerator
from agent1.generation.response_service import ResponseService
from agent1.agent import Agent

# Load only your own trusted saved classifier.
ensemble = IntentClassifierEnsemble.load(classifier_path)
# Leave the 4 GB GPU for Llama; sentence embeddings run on CPU.
ensemble.semantic_classifier.device = "cpu"
ensemble.semantic_classifier.embedding_model = None
knowledge = KnowledgeStore(knowledge_path)
generator = ResponseGenerator(adapter_dir)
responses = ResponseService(generator, knowledge)

def analyzer_llm(system_prompt, user_prompt):
    # V2 was trained to answer grounded questions, not to emit ActionPlan JSON.
    with generator.model.disable_adapter():
        return generator.generate([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ], max_new_tokens=512)

agent = Agent(
    ensemble_model=ensemble,
    router=IntentRouter(INTENT_TO_ACTION),
    analyzer=RequestAnalyzer(llm=analyzer_llm),
    context_builder=ContextBuilder(),
    response_service=responses,
)
print("Agent1 is ready. No training is performed.")

W1026 11:56:13.677000 9948 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Agent1 is ready. No training is performed.


In [3]:
def say(text):
    print("You:", text)
    answer = agent.handle_message(text)
    print("Agent1:", answer)
    print("Pending decision:", agent.last_pending_decision)
    print("Still waiting for arguments:", agent.state.has_pending_action())
    print("Response source:", responses.last_mode or "control flow")
    if agent.last_pending_error:
        print("Pending classifier error:", agent.last_pending_error)
    if agent.last_action_plan:
        print("Intent:", agent.last_action_plan.intent)
    print()
    return answer

def start_tracking():
    agent.reset()
    say("Where is my order?")
    assert agent.state.has_pending_action(), "Analyzer did not start tracking; inspect its plan before proceeding."


## Continue

In [4]:
start_tracking()
say('ORD-123')
print("expected_path:", agent.last_pending_decision == 'continue' and agent.state.has_pending_action() == False)

You: Where is my order?


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Agent1: Please provide one order number, such as ORD-123.
Pending decision: None
Still waiting for arguments: True
Response source: control flow
Intent: track_order

You: ORD-123
Agent1: Your order ORD-123 is shipped. Estimated delivery is 2026-09-23.
Pending decision: continue
Still waiting for arguments: False
Response source: v2
Intent: track_order

expected_path: True


## Abandon

In [5]:
start_tracking()
say('Never mind')
print("expected_path:", agent.last_pending_decision == 'abandon' and agent.state.has_pending_action() == False)

You: Where is my order?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Agent1: Please provide one order number, such as ORD-123.
Pending decision: None
Still waiting for arguments: True
Response source: control flow
Intent: track_order

You: Never mind
Agent1: Okay, I have stopped that request. No order was cancelled.
Pending decision: abandon
Still waiting for arguments: False
Response source: control flow

expected_path: True


## Switch

In [6]:
start_tracking()
say('Actually, what time does Central close on Saturday?')
print("expected_path:", agent.last_pending_decision == 'new_request' and agent.state.has_pending_action() == False)

You: Where is my order?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Agent1: Please provide one order number, such as ORD-123.
Pending decision: None
Still waiting for arguments: True
Response source: control flow
Intent: track_order

You: Actually, what time does Central close on Saturday?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Agent1: Would you like to know the store's opening hours or the delivery time for Saturday?
Pending decision: new_request
Still waiting for arguments: False
Response source: control flow
Intent: None

expected_path: True


## Ambiguous reply

In [7]:
start_tracking()
say('Yes')
print("expected_path:", agent.last_pending_decision == 'clarify' and agent.state.has_pending_action() == True)

You: Where is my order?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Agent1: Please provide one order number, such as ORD-123.
Pending decision: None
Still waiting for arguments: True
Response source: control flow
Intent: track_order

You: Yes
Agent1: Would you like to continue tracking your order, stop that request, or ask a different question?
Pending decision: clarify
Still waiting for arguments: True
Response source: control flow
Intent: track_order

expected_path: True


Expected: continuation executes the tracking tool once; abandonment does not cancel an actual order; switching answers Central's hours without looking up the old order; ambiguity preserves the pending task and asks one clarification.

The unit tests cover these transitions with controlled analyzer outputs, including invalid JSON and an ID embedded in a different request. These four live cases check the actual model. We defer task stacks, returning to abandoned tasks, general pronoun resolution and multi-action execution to later work.

After these paths work, move on to multiple actions. No V2 retraining or full benchmark rerun is required for this change.